In [ ]:
import numpy
import pandas

import matplotlib
import matplotlib.pyplot as plt
plt.style.use('mystyle.mplstyle')

import sbruceana

PATH_TO_SBRUCE = "/Users/triozzi/Analysis/numine/sbruceana/data/"

In [ ]:
FILE = "NuMI_Beam_Quality.txt"

VARS = [
  "NSLINA", "NSLINB", "NSLINC", "NSLIND", 
  "TRTGTD", "TR101D", 
  "pos1", "pos2",
  "width1", "width2"
]

df = pandas.read_csv(
  f"{PATH_TO_SBRUCE}{FILE}",
  names = VARS,
  delimiter = '\t',
  index_col = False
)

# horn current
df['horncurr'] = ((df['NSLINA'] - 0.01) / 0.9951) + \
                 ((df['NSLINB'] - (-0.14)) / 0.9957) + \
                 ((df['NSLINC'] - (-0.05)) / 0.9965) + \
                 ((df['NSLIND'] - (-0.07)) / 0.9945)

# POT
df['pot'] = df['TRTGTD']
df.loc[df['TRTGTD'] < 0.02, 'pot'] = df['TR101D']  # fall back to TR101D
df.loc[df['pot'] < 0., 'pot'] = 0.                 # clip negatives

pot_data = 2.5285722655952896e+19

In [ ]:
fig, axes = plt.subplots(figsize=(4*2, 3*3), ncols=2, nrows=3, layout='constrained')

### horn current
ax = axes[0, 0]

width = 0.05; bins = numpy.arange(-204, -194.5+width, width)
ax.hist(df['horncurr'], bins=bins, histtype='stepfilled')

ax = sbruceana.plotting.place_cut(ax, -196.4, True)
ax = sbruceana.plotting.place_cut(ax, -202, False)

ax.set(
  xlabel = 'horn current [mA]',
  ylabel = f'spills [#] / {width} mA',
  xlim   = (bins[0], bins[-1]),
)
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14) 

### POT
ax = axes[0, 1]

width = 0.5; bins = numpy.arange(-2, 65+width, width)
ax.hist(df['pot']*1e-12, bins=bins, histtype='stepfilled')

ax = sbruceana.plotting.place_cut(ax, 2, False)

ax.set(
  xlabel = 'POT [$\\times 10^{12}$]',
  ylabel = f'spills [#] / {width}',
  xlim   = (bins[0], bins[-1]),
)
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14) 

### beam pos x
ax = axes[1, 0]

width = 0.01; bins = numpy.arange(0, 1.2+width, width)
ax.hist(df['pos1'], bins=bins, histtype='stepfilled')

ax = sbruceana.plotting.place_cut(ax, 1, True)

ax.set(
  xlabel = 'beam $|x|$ position [mm]',
  ylabel = f'spills [#] / {width} mm',
  xlim   = (bins[0], bins[-1]),
)
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14) 

# beam pos y
ax = axes[1, 1]

width = 0.01; bins = numpy.arange(0, 1.2+width, width)
ax.hist(df['pos2'], bins=bins, histtype='stepfilled')

ax = sbruceana.plotting.place_cut(ax, 1, True)

ax.set(
  xlabel = 'beam $|y|$ position [mm]',
  ylabel = f'spills [#] / {width} mm',
  xlim   = (bins[0], bins[-1]),
)
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14) 

# beam width hor
ax = axes[2, 0]

width = 0.01; bins = numpy.arange(0, 2.5+width, width)
ax.hist(df['width1'], bins=bins, histtype='stepfilled')

ax = sbruceana.plotting.place_cut(ax, 0.57, False)
ax = sbruceana.plotting.place_cut(ax, 1.88, True)

ax.set(
  xlabel = 'horizontal width [mm]',
  ylabel = f'spills [#] / {width} mm',
  xlim   = (bins[0], bins[-1]),
)
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14) 

# beam width vert
ax = axes[2, 1]

width = 0.01; bins = numpy.arange(0, 2.5+width, width)
ax.hist(df['width2'], bins=bins, histtype='stepfilled')

ax = sbruceana.plotting.place_cut(ax, 0.57, False)
ax = sbruceana.plotting.place_cut(ax, 1.88, True)

ax.set(
  xlabel = 'vertical width [mm]',
  ylabel = f'spills [#] / {width} mm',
  xlim   = (bins[0], bins[-1]),
)
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14) 

fig.suptitle(f'Run2 10% NuMI data: {pot_data:.1e} POT', x=0.98, ha='right', fontsize=13, c='gray')

plt.show()
fig.savefig(f"beam_quality_prescaled.pdf", dpi=300)

In [ ]:
beam_quality_cuts = \
  (df['horncurr'] >= -202) & (df['horncurr'] <= -196.4) & \
  (df['pot']*1e-12 >= 2) & \
  (df['pos1'] <= 1) & (df['pos2'] <= 1) & \
  (df['width1'] >= 0.57) & (df['width1'] <= 1.88) & (df['width2'] >= 0.57) & (df['width2'] <= 1.88)

print('Initial number of spills:', len(df))
print('Number of spills passing beam quality cuts:', len(df[beam_quality_cuts]))
print('Fraction of spills passing the beam quality cuts [%]:', 100*(len(df[beam_quality_cuts]) / len(df)))

### Detector quality demonstration

In [ ]:
FILE = "NuMI_Detector_Quality_NSlc.txt"

df = pandas.read_csv(
  f"{PATH_TO_SBRUCE}{FILE}",
  names = ['run', 'nslc'],
  delimiter = ',',
)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3), layout='constrained')

ax.scatter(df.run, df.nslc, marker='.', c='black')
ax.scatter(9723, 33.23, marker='x', c='red')
ax.scatter(9643, 34.64, marker='x', c='red')

ax.axhline(32.31, lw=0.75, c='black')
ax.axhspan(31.44, 33.18, fc='C2', zorder=-3, label='3σ')
ax.axhspan(31.71, 32.91, fc='C1', zorder=-3, label='2σ')
ax.axhspan(32.01, 32.61, fc='C0', zorder=-3, label='1σ')

ax.set(
  title  = 'Run2 NuMI on-beam data',
  xlabel = 'DAQ run',
  ylabel = f'slices per event [#]',
  xlim   = (9500, 10000),
  ylim   = (30.75, 35),
)

handles, labels = ax.get_legend_handles_labels()
leg = ax.legend(handles[::-1], labels[::-1], ncol=3, fontsize=10, handlelength=1, columnspacing=0.5)

plt.show()
fig.savefig(f"detector_quality_example.pdf", dpi=300)